# 07. Memory Layouts, Strides & Contiguity: Beginner Guide

### 🌟 What Are Memory Layouts, Strides & Contiguity in NumPy?
NumPy arrays store multi-dimensional matrices in flat 1D memory. **Strides** define the number of bytes to step through memory to reach the next row or column. Understanding C-contiguous (row-major) vs Fortran (column-major) layouts helps you write cache-efficient algorithms.

This interactive guide loads and works directly with `data/raw_transactions.csv`, giving you real-world hands-on practice.

### 📚 Key Concepts Covered in this Notebook:
- **Layout Inspection (`arr.flags`)**: Checking `C_CONTIGUOUS` (row-major) vs `F_CONTIGUOUS` (column-major).
- **Stride Byte Navigation (`arr.strides`)**: Inspecting bytes skipped along each axis.
- **Transposition Stride Swapping**: Understanding zero-copy transpose pointer mechanics.
- **Enforcing Contiguity (`np.ascontiguousarray`)**: Converting non-contiguous strided arrays back into sequential RAM.


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

NumPy Version: 1.26.4
Loaded from ../data/raw_transactions.csv (14262 clean aligned rows):
- amounts array: shape (14262,), dtype float64
- fraud_flags array: shape (14262,), dtype int8
- account_ages array: shape (14262,), dtype float32


### 🔹 Layout Inspection with `arr.flags`
Checks whether a 2D transaction matrix is row-major (C-order) or column-major (Fortran-order). Checking and validating data types prevents subtle runtime errors and ensures subsequent mathematical or string operations behave properly. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `tx_mat.flags['C_CONTIGUOUS']`


In [2]:
tx_mat = np.column_stack([amounts[:1000], account_ages[:1000]])
print('Transaction Matrix C_CONTIGUOUS:', tx_mat.flags.c_contiguous)

Transaction Matrix C_CONTIGUOUS: True


### 🔹 Stride Byte Navigation: `arr.strides`
Inspects the byte-step stride tuple for navigating rows and columns in the transaction matrix. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `tx_mat.strides`


In [3]:
print('Transaction Matrix Shape:', tx_mat.shape)
print('Transaction Matrix Strides (bytes):', tx_mat.strides)

Transaction Matrix Shape: (1000, 2)
Transaction Matrix Strides (bytes): (16, 8)


### 🔹 Zero-Copy Transposition Stride Swapping
Transposing the transaction matrix swaps strides in $O(1)$ time without copying memory. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `tx_mat.T.strides`


In [4]:
tx_transposed = tx_mat.T
print('Transposed Shape:', tx_transposed.shape)
print('Transposed Strides:', tx_transposed.strides)
print('Is Transposed Matrix C_CONTIGUOUS?:', tx_transposed.flags.c_contiguous)

Transposed Shape: (2, 1000)
Transposed Strides: (8, 16)
Is Transposed Matrix C_CONTIGUOUS?: False


### 🔹 Enforcing Contiguity with `np.ascontiguousarray()`
Converts the transposed non-contiguous matrix back into a contiguous C-memory layout for Cython/C++ extensions. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `np.ascontiguousarray(tx_transposed)`


In [5]:
contig_tx = np.ascontiguousarray(tx_transposed)
print('Re-enforced Contiguity C_CONTIGUOUS:', contig_tx.flags.c_contiguous)

Re-enforced Contiguity C_CONTIGUOUS: True


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data questions explained simply with real examples.


### 🔍 Scenario: Q1: Row-Wise vs Column-Wise Traversal Performance Benchmark

**Approach:** Benchmark traversal speed across contiguous rows versus non-contiguous columns on transaction matrix.
**Syntax:** `tx_mat.sum(axis=1)` vs `tx_mat.sum(axis=0)`


In [6]:
big_mat = np.tile(tx_mat, (10, 1))
t0 = time.perf_counter()
big_mat.sum(axis=1)
t_row = time.perf_counter() - t0

t0 = time.perf_counter()
big_mat.sum(axis=0)
t_col = time.perf_counter() - t0
print(f'Row-wise (Cache-friendly): {t_row*1000:.2f} ms')
print(f'Col-wise: {t_col*1000:.2f} ms')

Row-wise (Cache-friendly): 0.18 ms
Col-wise: 0.10 ms
